# M2 T2 Practical — K-means and DBSCAN

Revision: 11 September 2026

## Learning outcomes

- Fit and interpret K-means clusters and their arbitrary labels.
- Use inertia and silhouette critically alongside geometry.
- Identify DBSCAN core, border and noise observations.
- Investigate eps, min_samples and feature scaling with controlled comparisons.
- Recommend a clustering method using application requirements and limitations.

## How to work and submit

Plan for a two-hour session: about 90 minutes of core work and 30 minutes for discussion, debugging and checking. Timings beside questions are estimates, not deadlines. Setup and plotting support are supplied; focus on the modelling ideas.

Run cells in order. Replace ... in TODO cells and write in each answer cell. Make a prediction before running an experiment; keep it even if the result surprises you, then explain the difference. If a cell fails, ask about the first error and retain your genuine attempt.

For every explanation, cite a number, observation or intermediate value from your own run, explain what it means, and justify your conclusion. Concise supported answers are enough. Different justified choices may be valid. Save your notebook with outputs and written answers. Supplied code alone is not a completed response.

The five core questions are assessed using the course's 50% completion/genuine attempt and 50% demonstrated understanding criteria. Optional extensions are not required. No new package installation or external dataset download is needed in the course Python environment (NumPy, pandas, matplotlib and scikit-learn).

## Preparation and links to the topic

Read [M2 T2 overview](https://learn.adelaide.edu.au/courses/30362/pages/module-2-topic-2-overview), K-means and DBSCAN in that topic, then [Evaluating Unsupervised Learning](https://learn.adelaide.edu.au/courses/30362/pages/evaluating-unsupervised-learning). Q1–2 connect fitting to inertia and silhouette; Q3 tests density rules and scaling; Q4–5 connect geometry to application decisions. The tutorial covers hand assignment/centroid updates; this practical tests larger-data interpretation. Silhouette/BIC derivations are not required.

These are synthetic observations with no verified customer labels. Any customer or service interpretation is an illustrative scenario.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.cluster import DBSCAN, KMeans
from sklearn.datasets import make_blobs, make_moons
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
X_raw, _ = make_blobs(
    n_samples=260, centers=[(-4,-1), (0,3), (4,-1)],
    cluster_std=[0.8, 1.0, 0.7], random_state=12
)
X_raw = np.vstack([X_raw, [[-7,5], [7,5], [0,-6], [7,-4]]])
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

def show_clusters(labels, title, centers=None):
    plt.figure(figsize=(5.8, 3.8))
    plt.scatter(X[:,0], X[:,1], c=labels, cmap="tab10", edgecolor="k", s=30)
    if centers is not None:
        plt.scatter(centers[:,0], centers[:,1], c="black", marker="X", s=170)
    plt.title(title); plt.xlabel("Scaled feature 1"); plt.ylabel("Scaled feature 2"); plt.show()

plt.scatter(X[:,0], X[:,1], edgecolor="k", s=30)
plt.title("Unlabelled scaled data"); plt.show()

## Question 1 — Establish a baseline (10 minutes)

Complete k. We use init="k-means++" to choose spread-out starting centroids. n_init=10 fits ten initialisations and retains the lowest-inertia result, reducing dependence on one unlucky start.

In your written answer, describe the fitted cluster shapes and sizes, explain what the centroids and numerical labels mean, and state what K-means does with the isolated observations.

In [ ]:
# TODO 1: use three clusters.
kmeans = KMeans(init="k-means++", n_clusters=3, n_init=10, random_state=RANDOM_STATE)
k_labels = kmeans.fit_predict(X)
show_clusters(k_labels, "K-means with k=3", kmeans.cluster_centers_)
print("Inertia:", round(kmeans.inertia_, 2))
print("Cluster sizes:", np.bincount(k_labels))

### Write your answer — Question 1

The fitted k-means result has three places where the data points are fairly dense, grouped around their centroid. The cluster size are 88, 88, and 88 respectively, meaning they are evenly distributed here. The inertia is 73.98.

The centroid of a cluster are the mean position of all the data points that belongs to that cluster. The numerical label are only identifiers and doesn't do anything other than identifying the clusters.

K-mean assigns every data point to the cluster whose centroid is the nearest. Therefore, there is no chance of an isolated observation to be marked as an outlier. They will be added to the nearest centroid's cluster.

## Question 2 — Choose k and test the evidence (15 minutes)

Compare k=2 through 6 using the supplied runner. Choose a defensible k and cite one inertia comparison and one silhouette comparison. Explain why declining inertia alone cannot select k. State one further check needed before treating these as customer segments.

In [ ]:
rows = []
# TODO 2: use k values 2, 3, 4, 5 and 6. Remember range stops before its second argument.
for k in range(2, 7):
    model = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels = model.fit_predict(X)
    rows.append((k, model.inertia_, silhouette_score(X, labels)))
scores = pd.DataFrame(rows, columns=["k", "inertia", "silhouette"])
display(scores)
fig, ax = plt.subplots(1,2,figsize=(10,3.5))
ax[0].plot(scores.k, scores.inertia, marker="o"); ax[0].set_title("Inertia")
ax[1].plot(scores.k, scores.silhouette, marker="o"); ax[1].set_title("Silhouette")
for a in ax: a.set_xlabel("k")
plt.show()

### Write your answer — Question 2

When k=2, The inertia is at its highest and silhouette is relatively low at 0.50. when k = 3, inertia takes a massive dip while silhouette also spikes significantly. After k=3, each subsequent increase of k causes steady decrease of inertia which can't be called significant. The silhouette, on the other hand, starts taking a major dips with each increase. 

Therefore, the best k to choose is 3 since it introduces the most dip in the inertia from the previous, and the highest silhouette.

Inertia alone can't choose k because inertia always decreases or stays the same as k increases. Therefore, if we took only inertia into consideration, larger k value would keep looking better even if they split meaningful clusters into smaller clusters.

## Question 3 — Density and a controlled intervention (20 minutes)

Complete the masks and eps candidates below. Before running, predict how increasing eps changes core/noise counts. Compare the observed counts with your prediction. Then hold eps=0.32 fixed and compare min_samples=3 and 12 using dbscan_summary. Explain core, border and noise using the results, and why noise is not proof of an erroneous record. Finally predict what happens if feature 1 is measured in units 100 times larger without rescaling; run the supplied comparison and explain it.

In [ ]:
def dbscan_summary(eps, min_samples=5):
    model = DBSCAN(eps=eps, min_samples=min_samples).fit(X)
    labels = model.labels_

    # TODO 3: create Boolean masks for core, noise and border points.
    core_mask = np.zeros(len(X), dtype=bool)
    core_mask[model.core_sample_indices_] = True
    noise_mask = (labels == -1)
    border_mask = (~core_mask & ~noise_mask)

    clusters = len(set(labels)) - (1 if -1 in labels else 0)
    return labels, core_mask, border_mask, noise_mask, clusters

dbscan_rows = []
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharex=True, sharey=True)
# TODO 4: compare eps values 0.20, 0.32 and 0.50.
for axis, eps in zip(axes, [0.20, 0.32, 0.50]):
    labels, core, border, noise, clusters = dbscan_summary(eps)
    dbscan_rows.append({
        "eps": eps, "clusters": clusters,
        "core": int(core.sum()), "border": int(border.sum()),
        "noise": int(noise.sum()),
    })
    axis.scatter(X[core, 0], X[core, 1], c=labels[core], cmap="tab10", vmin=0, vmax=max(1, clusters-1), s=34, label="core")
    axis.scatter(X[border, 0], X[border, 1], c=labels[border], cmap="tab10", vmin=0, vmax=max(1, clusters-1), marker="D", s=42, edgecolor="black", label="border")
    axis.scatter(X[noise, 0], X[noise, 1], c="black", marker="x", s=42, label="noise")
    axis.set_title(f"eps={eps}: {clusters} clusters")
    axis.set_xlabel("Scaled feature 1")
axes[0].set_ylabel("Scaled feature 2")
axes[-1].legend(loc="best")
plt.tight_layout()
plt.show()
display(pd.DataFrame(dbscan_rows))


# TODO: change only min_samples in these two calls.
for minimum in [3, 12]:
    labels, core, border, noise, count = dbscan_summary(0.32, min_samples=minimum)
    print(minimum, count, int(core.sum()), int(border.sum()), int(noise.sum()))

# Supplied unit-change experiment: fixed eps/min_samples, no new fit choices.
X_units = X.copy(); X_units[:, 0] *= 100
for label, features in [("original scaled", X), ("changed units, unscaled", X_units), ("changed units, rescaled", StandardScaler().fit_transform(X_units))]:
    labs = DBSCAN(eps=0.32, min_samples=5).fit_predict(features)
    print(label, "clusters", len(set(labs)-{-1}), "noise", int((labs == -1).sum()))

### Write your answer — Question 3

As eps increases, the number of observations that count as neighbours decrease. Subsequently, the number of core point increase resulting in the decrease of border points and noise points. If eps becomes too large, then two different clusters can merge.

The results match the first part of that prediction. At eps=0.20 there are 195 core, 25 border and 44 noise observations. When eps=0.32 this changes to 240 core, 15 border and 9 noise, and at eps=0.50 there are 256 core, 4 border and 4 noise. As eps increases, so does core point count while border and noise point count goes down. But all three settings still produce 3 clusters, so the tested increase in eps did not yet merge the three main groups.

A core observation has enough nearby observations within eps to satisfy min_samples. A border observation belongs to a cluster because it is reachable from a core observation but does not itself have enough neighbours to be core. A noise observation has label -1 because it is not density-reachable from a cluster under the chosen settings. When eps=0.32, lowering min_samples to 3 gives 254 core, 5 border and 5 noise, while raising it to 12 gives 199 core, 43 border and 22 noise. This shows that a stricter density requirement makes fewer points core and more points border or noise.

Noise is not proof that a record is wrong. It only means that the record lies in a region that is not dense enough under the current eps and min_samples which is a rare but valid observation could therefore be labelled as noise.

If feature 1 is multiplied by 100 without rescaling, distances in that feature should dominate DBSCAN and badly change the result. What happens is that the original scaled data produce 3 clusters and 9 noise observations, while the changed-units unscaled data produce 1 cluster and 258 noise observations. After applying StandardScaler again, the result returns to 3 clusters and 9 noise, showing why distance-based clustering needs comparable feature scales.

In [ ]:
X_moons_raw, _ = make_moons(n_samples=320, noise=0.08, random_state=RANDOM_STATE)
X_moons = StandardScaler().fit_transform(X_moons_raw)
def clustering_evidence(features, labels):
    keep = labels != -1
    groups = len(set(labels[keep]))
    # Silhouette requires 2 <= number of labels < number of scored observations.
    score = silhouette_score(features[keep], labels[keep]) if 2 <= groups < int(keep.sum()) else np.nan
    return groups, int((~keep).sum()), score
def compare_moons(eps_values):
    rows=[]; fig, axes=plt.subplots(1, len(eps_values)+1, figsize=(14,3.5))
    models=[('K-means', KMeans(n_clusters=2,n_init=10,random_state=42))]+[(f'DBSCAN eps={e}', DBSCAN(eps=e,min_samples=5)) for e in eps_values]
    for ax,(name,model) in zip(axes,models):
        labels=model.fit_predict(X_moons)
        groups,noise,score=clustering_evidence(X_moons,labels)
        rows.append((name,groups,noise,score))
        keep = labels != -1
        ax.scatter(X_moons[keep,0],X_moons[keep,1],c=labels[keep],cmap='tab10',s=12)
        if (~keep).any():
            ax.scatter(X_moons[~keep,0],X_moons[~keep,1],c='black',marker='x',s=18,label='noise')
            ax.legend(loc='best')
        ax.set_title(name); ax.set_xlabel('Scaled feature 1')
    axes[0].set_ylabel('Scaled feature 2');plt.tight_layout();plt.show()
    return pd.DataFrame(rows,columns=['method','clusters','noise','silhouette_non_noise'])

## Question 4 — Investigate curved groups (25 minutes)

The application defines each connected curved band as a useful group and permits some observations to remain unassigned. Before fitting, predict which algorithm assumptions suit this requirement. Complete a comparison of eps=0.15, 0.28 and 0.60; recommend one result using geometry, cluster/noise counts and silhouette. Explain any disagreement between the score and your recommendation. A missing silhouette is a valid outcome: explain why it is unavailable. When noise differs, note that scores describe different subsets.

In [ ]:
# TODO: supply the three candidate eps values.
moon_results = compare_moons([0.15,0.28,0.60])
display(moon_results)

### Write your answer — Question 4
DBSCAN should suit this requirement better than K-means because DBSCAN can follow connected, non-spherical shapes and can leave observations as noise. K-means instead forms clusters around centroids, so it is more suited to compact groups.

The recommended eps is 0.28. It finds exactly 2 clusters with 0 noise and the plot follows the two connected curved moon-shaped bands. Its silhouette score is about 0.379. When eps=0.15, DBSCAN fragments the data into 19 clusters and labels 52 observations as noise, so the neighbourhood is too small for the application's desired two curved groups. When eps=0.60, DBSCAN merges everything into 1 cluster, which is also unsuitable.

K-means gives 2 clusters, 0 noise and a higher silhouette of about 0.494, but its boundaries cut the curved geometry into centroid-based regions instead of following the two bands. Therefore, the higher silhouette does not make K-means the better choice for the stated application. Silhouette rewards within-cluster closeness and between-cluster separation, and that criterion does not necessarily match the desired curved connectivity.

The silhouette for DBSCAN when eps=0.60 is missing because there is only one non-noise cluster, while silhouette requires at least two clusters. Also, when noise differs between DBSCAN settings, the silhouette values are calculated on different subsets of observations. For example, eps=0.15 excludes 52 noise points from its score.


## Question 5 — Make a defensible recommendation (20 minutes)

Two teams want to reuse your results. Team A needs exactly three compact service territories and must assign every record. Team B wants connected regions and can send unusual records for manual review. Recommend a method for each, using one result from this notebook. For Team B, propose a concrete check before deployment when a valid region has lower density. State how duplicated records could affect density-based clustering and what data checks you would perform. Do not describe fitted labels as real identities.

### Write your answer — Question 5

For Team A, K-means should be used with k=3. The team requires exactly three compact territories and every record must be assigned. In Question 1, K-means produced exactly 3 clusters of 88 observations each, and in Question 2, k=3 had the strongest silhouette of the tested values at about 0.714. K-means also assigns isolated observations rather than leaving them as noise, which Team A needs.

For Team B, I would use DBSCAN because the team cares about connected regions and is allowed to send unusual observations for manual review. On the curved-data experiment, DBSCAN with eps=0.28 recovered the 2 connected curved bands, whereas K-means did not follow that geometry. DBSCAN can also label low-density observations as noise when appropriate.

Before deployment, I would specifically test the model on representative examples from any valid lower-density region. I would inspect its neighbour counts and rerun a controlled range of eps and min_samples values to check whether that valid region is incorrectly fragmented or marked as noise. The final parameters should be checked with domain experts or known valid examples, rather than assuming that all low-density observations are invalid.

Duplicated records can artificially increase local density in DBSCAN. Enough duplicates could turn a border or noise point into a core point, create an artificial cluster, or help connect regions that should remain separate. I would check for exact duplicates and near-duplicates, examine duplicate rates by region, determine whether repeated rows are legitimate repeated events, and deduplicate or otherwise handle them before fitting when they are accidental.

## Optional extensions

Complete these only after the core. Each is worth up to 5 bonus marks under the course policy; omitting either loses no core marks.

- Repeat K-means across five seeds and report stability without comparing arbitrary label numbers directly.
- Repeat the moons comparison at a second noise level and assess whether your recommendation still holds.